In [40]:
from dotenv import load_dotenv
import json
import whisper
from pyannote.audio import Pipeline
from pyannote_whisper.utils import diarize_text
import os
from pathlib import Path
from pydub import AudioSegment
import shutil, errno
import pandas as pd

In [24]:
def extension(file):
    return Path(file).suffix.replace('.', '')

def copydir(src, dst):
    try:
        shutil.copytree(src, dst)
    except OSError as exc: # python >2.5
        if exc.errno in (errno.ENOTDIR, errno.EINVAL):
            shutil.copy(src, dst)
        else: raise

In [25]:
file = 'data/Parker Sams DJNF profile.m4a'

In [26]:
output_name = Path(file).stem.replace(' ', '_')
copyanything('web', output_name)

In [27]:
raw_audio = AudioSegment.from_file(file, format=extension(file))
formatted_file = f'{output_name}/audio.mp3'
raw_audio.export(formatted_file, format='mp3')

<_io.BufferedRandom name='Parker_Sams_DJNF_profile/audio.mp3'>

In [28]:
load_dotenv('openai.env')

True

In [29]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=os.getenv('PYANNOTE'))

In [8]:
diarization_result = pipeline(formatted_file)

/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is

In [9]:
model = whisper.load_model("base")

In [10]:
asr_result = model.transcribe(formatted_file)

/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [11]:
final_result = diarize_text(asr_result, diarization_result)

In [47]:
speaker_matches = {list(final_result[i])[0].start : list(final_result[i])[1] if list(final_result[i])[1] is not None else list(final_result[i-1])[1] for i in range(0, len(final_result))}

In [48]:
switches = list(speaker_matches.keys())
for i in range(0, len(asr_result['segments'])):
    curr_start = asr_result['segments'][i]['start']
    for j in range(0, len(switches)):
        if curr_start < switches[j]:
            switch = switches[j-1]
            break
    asr_result['segments'][i]['speaker'] = speaker_matches[switch]

In [49]:
result = asr_result

In [50]:
with open(f'{output_name}/transcript.json', 'w') as out_path:  
    json.dump(result, out_path)